In [1]:
from datasets import load_dataset, Dataset, DatasetDict, load_from_disk
import pandas as pd

b:\stacks\Machine_learning_projects\Video_Summarization_Translation\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
ds = load_dataset("retkowski/ytseg", "text")
ds.save_to_disk("../datasets/ytseg-dataset")

In [2]:

def prepare_ytseg_data(split_name="train"):
    # 1. Load the 'text' subset of the dataset
    ds = load_from_disk("../datasets/ytseg-dataset")[split_name]
    
    processed_data = []
    
    for example in ds:
        # 'text_ref' contains the list of sentences
        # 'target_binary_ref' contains the 0/1 labels for each sentence
        sentences = example['text_ref']
        labels = [int(x) for x in example['target_binary_ref'][2:]] # Convert string/list to ints
        
        for i in range(len(sentences)):
            # Target Sentence
            target = sentences[i]
            
            # Context window: Previous sentence + Current + Next sentence
            # This is crucial for segmentation/classification tasks
            prev_sent = sentences[i-1] if i > 0 else ""
            next_sent = sentences[i+1] if i < len(sentences)-1 else ""
            
            # Option A: Simple concatenation for the 'text' field
            # Option B: Use [SEP] tokens (handled by tokenizer.text_pair)
            context = f"{prev_sent} [SEP] {target} [SEP] {next_sent}" # or however you prefer
            
            processed_data.append({
                "sentence": target,
                "context_prev": prev_sent,
                "context_next": next_sent,
                "label": labels[i]
            })
            
    return Dataset.from_pandas(pd.DataFrame(processed_data))

#Process all splits
train_ds = prepare_ytseg_data("train")
val_ds = prepare_ytseg_data("validation")

In [3]:
from transformers import AutoTokenizer

model_id = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_fn(batch):
    # We provide the target sentence and the combined context as a pair
    # This uses DeBERTa's ability to handle two distinct text segments
    return tokenizer(
        batch["sentence"],
        [f"{p} [SEP] {n}" for p, n in zip(batch["context_prev"], batch["context_next"])],
        truncation=True,
        padding="max_length",
        max_length=256 # Adjust based on GPU memory
    )

tokenized_train = train_ds.map(tokenize_fn, batched=True)
tokenized_val = val_ds.map(tokenize_fn, batched=True)

b:\stacks\Machine_learning_projects\Video_Summarization_Translation\.venv\lib\site-packages\transformers\convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Map: 100%|██████████| 286753/286753 [00:31<00:00, 8998.60 examples/s] 


In [4]:
print(tokenizer.sep_token)

[SEP]


In [5]:
tokenized_train = tokenized_train.shuffle().select(range(30000))
tokenized_val = tokenized_val.shuffle().select(range(3000))

In [6]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

args = TrainingArguments(
    output_dir="./deberta-ytseg-finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=20,
    weight_decay=0.01,
    push_to_hub=False,
    fp16=True, # Use Mixed Precision for DeBERTa (highly recommended)
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

trainer.train()

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,F1
1,0.213700,0.230390,0.000000
2,0.221200,0.193672,0.120000
3,0.161600,0.197590,0.125786
4,0.141800,0.218038,0.258065
5,0.187600,0.210131,0.206061
6,0.165500,0.210802,0.000000
7,0.100300,0.347200,0.197628
8,0.089600,0.239320,0.202128
9,0.139800,0.216917,0.185185
10,0.102700,0.237213,0.126582


KeyboardInterrupt: 

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(, num_labels=2)